# Data Wrangling - Pantau Pasar
**Sumber Data**:
- PIHPS Nasional (hargapangan.id) 2022 s/d Apr 2026
- SP2KP Kemendag Jan s/d Apr 2026

**Alur**: Gathering -> Assessing -> Cleaning

In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)

---
## 1. Gathering Data

### 1.1 Load Dataset PIHPS (5 File per Tahun)

In [83]:
PIHPS_FILES = {
    2022: "data/raw/Tabel Harga Berdasarkan Daerah_2022.xlsx",
    2023: "data/raw/Tabel Harga Berdasarkan Daerah_2023.xlsx",
    2024: "data/raw/Tabel Harga Berdasarkan Daerah_2024.xlsx",
    2025: "data/raw/Tabel Harga Berdasarkan Daerah_2025.xlsx",
    2026: "data/raw/Tabel Harga Berdasarkan Daerah_2026_jan-apr.xlsx",
}

raw_pihps = {}
for year, fname in PIHPS_FILES.items():
    raw_pihps[year] = pd.read_excel(fname)
    print(f"[{year}] {fname} | shape: {raw_pihps[year].shape}")

[2022] data/raw/Tabel Harga Berdasarkan Daerah_2022.xlsx | shape: (31, 262)
[2023] data/raw/Tabel Harga Berdasarkan Daerah_2023.xlsx | shape: (31, 262)
[2024] data/raw/Tabel Harga Berdasarkan Daerah_2024.xlsx | shape: (31, 264)
[2025] data/raw/Tabel Harga Berdasarkan Daerah_2025.xlsx | shape: (31, 263)
[2026] data/raw/Tabel Harga Berdasarkan Daerah_2026_jan-apr.xlsx | shape: (31, 88)


### 1.2 Load Dataset SP2KP

In [84]:
raw_sp2kp = pd.read_csv("data/raw/Tabulasi SP2KP.csv", encoding="utf-16", sep="	")
print(f"SP2KP | shape: {raw_sp2kp.shape}")
raw_sp2kp.head(3)

SP2KP | shape: (7830, 65)


,cekNo,Kode Wilayah,Provinsi,Kabupaten Kota,Komoditas,...,24/04/2026,27/04/2026,28/04/2026,29/04/2026,30/04/2026
0,1.0,1101,Aceh,Kab. Aceh Selatan,Bawang Merah,...,35.00,35.00,35.00,35.00,31.667
1,2.0,1101,Aceh,Kab. Aceh Selatan,Bawang Putih Honan,...,36.00,36.00,36.00,36.00,36.000
2,3.0,1101,Aceh,Kab. Aceh Selatan,Beras Medium,...,14.35,14.35,14.35,14.35,14.350


### 1.3 Preview Sampel Data Mentah PIHPS

In [85]:
raw_pihps[2022].head()

,No,Komoditas (Rp),03/ 01/ 2022,04/ 01/ 2022,05/ 01/ 2022,...,26/ 12/ 2022,27/ 12/ 2022,28/ 12/ 2022,29/ 12/ 2022,30/ 12/ 2022
0,I,Beras,"11,750","11,750","11,750",...,"12,600","12,600","12,600","12,600","12,600"
1,1,Beras Kualitas Bawah I,"10,750","10,750","10,700",...,"11,450","11,450","11,450","11,450","11,500"
2,2,Beras Kualitas Bawah II,"10,400","10,450","10,400",...,"11,200","11,150","11,200","11,200","11,200"
3,3,Beras Kualitas Medium I,"11,750","11,750","11,750",...,"12,600","12,600","12,600","12,600","12,600"
4,4,Beras Kualitas Medium II,"11,550","11,550","11,550",...,"12,400","12,400","12,400","12,400","12,450"


---
## 2. Assessing Data

### 2.1 Struktur & Tipe Data per File

In [86]:
for year, df in raw_pihps.items():
    date_cols = df.columns[2:]
    print(f'--- {year} ---')
    print(f'  Dimensi      : {df.shape[0]} baris x {df.shape[1]} kolom')
    print(f'  Periode      : {date_cols[0].strip()} s/d {date_cols[-1].strip()}')
    print(f'  Tipe kolom   : No={df["No"].dtype}, Komoditas={df["Komoditas (Rp)"].dtype}, Harga={df.iloc[:,2].dtype}')
    print()

--- 2022 ---
  Dimensi      : 31 baris x 262 kolom
  Periode      : 03/ 01/ 2022 s/d 30/ 12/ 2022
  Tipe kolom   : No=object, Komoditas=object, Harga=object

--- 2023 ---
  Dimensi      : 31 baris x 262 kolom
  Periode      : 02/ 01/ 2023 s/d 29/ 12/ 2023
  Tipe kolom   : No=object, Komoditas=object, Harga=object

--- 2024 ---
  Dimensi      : 31 baris x 264 kolom
  Periode      : 01/ 01/ 2024 s/d 31/ 12/ 2024
  Tipe kolom   : No=object, Komoditas=object, Harga=object

--- 2025 ---
  Dimensi      : 31 baris x 263 kolom
  Periode      : 01/ 01/ 2025 s/d 31/ 12/ 2025
  Tipe kolom   : No=object, Komoditas=object, Harga=object

--- 2026 ---
  Dimensi      : 31 baris x 88 kolom
  Periode      : 01/ 01/ 2026 s/d 30/ 04/ 2026
  Tipe kolom   : No=object, Komoditas=object, Harga=object



### 2.2 Identifikasi Missing Values

In [87]:
col_tahun     = 'Tahun'
col_nan       = 'NaN'
col_dash      = 'Dash(-)'
col_total     = 'Total Sel'
col_missing   = 'Missing%'

print('=== Missing Values per File PIHPS ===')
print(f'{col_tahun:<8} {col_nan:>8} {col_dash:>10} {col_total:>12} {col_missing:>10}')
print('-' * 55)
for year, df in raw_pihps.items():
    data_only   = df.iloc[:, 2:]
    nan_count   = data_only.isna().sum().sum()
    dash_count  = (data_only == '-').sum().sum()
    total       = data_only.size
    missing_pct = (nan_count + dash_count) / total * 100
    print(f'{year:<8} {nan_count:>8} {dash_count:>10} {total:>12} {missing_pct:>9.1f}%')

=== Missing Values per File PIHPS ===
Tahun         NaN    Dash(-)    Total Sel   Missing%
-------------------------------------------------------
2022            0        191         8060       2.4%
2023            0          0         8060       0.0%
2024            0          0         8122       0.0%
2025            0          0         8091       0.0%
2026            0          0         2666       0.0%


### 2.3 Identifikasi Baris Header Grup (Nomor Romawi)

In [88]:
ROMAN = ['I','II','III','IV','V','VI','VII','VIII','IX','X']

df_sample   = raw_pihps[2022]
header_rows = df_sample[df_sample['No'].isin(ROMAN)]
sub_rows    = df_sample[~df_sample['No'].isin(ROMAN)]

print('Baris header grup (akan dihapus):')
print(header_rows[['No', 'Komoditas (Rp)']].to_string(index=False))
print(f'\nJumlah sub-komoditas valid: {len(sub_rows)}')

Baris header grup (akan dihapus):
  No Komoditas (Rp)
   I          Beras
  II    Daging Ayam
 III    Daging Sapi
  IV     Telur Ayam
   V   Bawang Merah
  VI   Bawang Putih
 VII    Cabai Merah
VIII    Cabai Rawit
  IX  Minyak Goreng
   X     Gula Pasir

Jumlah sub-komoditas valid: 21


### 2.4 Identifikasi Masalah Format Kolom Tanggal & Harga

In [89]:
df_sample      = raw_pihps[2022]
first_date_col = df_sample.columns[2]
sample_value   = df_sample.iloc[3, 2]

print(f'Format kolom tanggal (raw)  : "{first_date_col}"')
print(f'Format nilai harga (raw)    : "{sample_value}", tipe: {type(sample_value).__name__}')
print()
print('Masalah yang teridentifikasi:')
print('  1. Kolom tanggal memiliki spasi  -> perlu strip() dan pd.to_datetime()')
print('  2. Nilai harga bertipe string    -> perlu replace koma dan konversi ke float')
print('  3. Missing value berupa tanda -  -> perlu replace ke NaN sebelum interpolasi')
print('  4. Nama komoditas ada whitespace -> perlu str.strip()')
print('  5. Format Wide -> perlu diubah ke Long dengan pd.melt()')

Format kolom tanggal (raw)  : "03/ 01/ 2022"
Format nilai harga (raw)    : "11,750", tipe: str

Masalah yang teridentifikasi:
  1. Kolom tanggal memiliki spasi  -> perlu strip() dan pd.to_datetime()
  2. Nilai harga bertipe string    -> perlu replace koma dan konversi ke float
  3. Missing value berupa tanda -  -> perlu replace ke NaN sebelum interpolasi
  4. Nama komoditas ada whitespace -> perlu str.strip()
  5. Format Wide -> perlu diubah ke Long dengan pd.melt()


### 2.5 Konsistensi Nama Komoditas Antar Tahun

In [90]:
base_commodities = raw_pihps[2022]['Komoditas (Rp)'].tolist()

print('Konsistensi nama komoditas:')
for year, df in raw_pihps.items():
    match = df['Komoditas (Rp)'].tolist() == base_commodities
    print(f'  {year}: {"KONSISTEN" if match else "TIDAK KONSISTEN"}')

Konsistensi nama komoditas:
  2022: KONSISTEN
  2023: KONSISTEN
  2024: KONSISTEN
  2025: KONSISTEN
  2026: KONSISTEN


### 2.6 Assessing SP2KP

In [91]:
print('=== SP2KP ===')
print(f'Dimensi        : {raw_sp2kp.shape}')
print(f'Kolom awal     : {raw_sp2kp.columns[:6].tolist()}')
print(f'Jumlah NaN     : {raw_sp2kp.isna().sum().sum()}')
print(f'Provinsi unik  : {raw_sp2kp.iloc[:,2].nunique()}')
print(f'Kabupaten unik : {raw_sp2kp.iloc[:,3].nunique()}')
print(f'Komoditas unik : {raw_sp2kp.iloc[:,4].nunique()}')
print()
print('Daftar komoditas SP2KP:')
print(raw_sp2kp.iloc[:,4].unique())

=== SP2KP ===
Dimensi        : (7830, 65)
Kolom awal     : ['cekNo', 'Kode Wilayah', 'Provinsi', 'Kabupaten Kota', 'Komoditas ', 'HET/HA ']
Jumlah NaN     : 7837
Provinsi unik  : 38
Kabupaten unik : 514
Komoditas unik : 17

Daftar komoditas SP2KP:
['Bawang Merah' 'Bawang Putih Honan' 'Beras Medium' 'Beras Premium'
 'Cabai Merah Keriting' 'Daging Ayam Ras' 'Garam Halus' 'Gula Pasir Curah'
 'Ikan Kembung' 'Minyak Goreng Sawit Curah'
 'Minyak Goreng Sawit Kemasan Premium' 'Minyakita' 'Telur Ayam Ras'
 'Tepung Terigu' 'Cabai Merah Besar' 'Cabai Rawit Merah'
 'Daging Sapi Paha Belakang']


---
## 3. Cleaning Data

### 3.1 Cleaning PIHPS - Fungsi Utama

In [92]:
ROMAN = ['I','II','III','IV','V','VI','VII','VIII','IX','X']

def clean_pihps(df):
    # Hapus baris header grup (nomor romawi)
    df = df[~df['No'].isin(ROMAN)].copy()

    # Bersihkan whitespace pada nama komoditas
    df['Komoditas (Rp)'] = df['Komoditas (Rp)'].str.strip()

    # Bersihkan whitespace pada nama kolom tanggal
    df.columns = ['No', 'Komoditas'] + [c.replace(' ', '') for c in df.columns[2:]]

    # Melt: wide -> long
    df_long = df.melt(
        id_vars=['Komoditas'],
        value_vars=df.columns[2:],
        var_name='tanggal',
        value_name='harga'
    )

    # Ganti tanda '-' dengan NaN
    df_long['harga'] = df_long['harga'].replace('-', np.nan)

    # Konversi harga dari string ke float
    df_long['harga'] = df_long['harga'].astype(str).str.replace(',', '', regex=False)
    df_long['harga'] = pd.to_numeric(df_long['harga'], errors='coerce')

    # Konversi tanggal ke datetime
    df_long['tanggal'] = pd.to_datetime(df_long['tanggal'], format='%d/%m/%Y')

    # Urutkan dan reset index
    df_long = df_long.sort_values(['Komoditas', 'tanggal']).reset_index(drop=True)

    return df_long

print('Fungsi clean_pihps() siap digunakan.')

Fungsi clean_pihps() siap digunakan.


### 3.2 Terapkan Cleaning ke Semua File & Gabungkan

In [93]:
cleaned_dfs = []
for year, df in raw_pihps.items():
    cleaned = clean_pihps(df)
    cleaned_dfs.append(cleaned)
    print(f'[{year}] setelah cleaning: {cleaned.shape[0]:,} baris, {cleaned["Komoditas"].nunique()} komoditas')

df_pihps = pd.concat(cleaned_dfs, ignore_index=True)
df_pihps = df_pihps.sort_values(['Komoditas', 'tanggal']).reset_index(drop=True)

# Tandai sumber data untuk traceability
df_pihps['sumber'] = 'PIHPS'

print(f'\nDataset PIHPS gabungan: {df_pihps.shape[0]:,} baris x {df_pihps.shape[1]} kolom')

[2022] setelah cleaning: 5,460 baris, 21 komoditas
[2023] setelah cleaning: 5,460 baris, 21 komoditas
[2024] setelah cleaning: 5,502 baris, 21 komoditas
[2025] setelah cleaning: 5,481 baris, 21 komoditas
[2026] setelah cleaning: 1,806 baris, 21 komoditas

Dataset PIHPS gabungan: 23,709 baris x 4 kolom


### 3.3 Tangani Missing Values PIHPS (Interpolasi)

In [94]:
print(f'Missing values sebelum interpolasi: {df_pihps["harga"].isna().sum()}')

# Interpolasi linear per komoditas agar tidak mencampur pola antar komoditas
df_pihps['harga'] = (
    df_pihps
    .groupby('Komoditas')['harga']
    .transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
)

print(f'Missing values setelah interpolasi : {df_pihps["harga"].isna().sum()}')

Missing values sebelum interpolasi: 131
Missing values setelah interpolasi : 0


### 3.4 Validasi Dataset PIHPS Final

In [95]:
print('=== Validasi Dataset PIHPS ===')
print(f'Shape           : {df_pihps.shape}')
print(f'Rentang tanggal : {df_pihps["tanggal"].min().date()} s/d {df_pihps["tanggal"].max().date()}')
print(f'Jumlah komoditas: {df_pihps["Komoditas"].nunique()}')
print(f'Missing values  : {df_pihps.isna().sum().sum()}')
print()
print('Tipe data:')
print(df_pihps.dtypes)
print()
print('Daftar komoditas:')
for k in sorted(df_pihps['Komoditas'].unique()):
    print(f'  - {k}')

=== Validasi Dataset PIHPS ===
Shape           : (23709, 4)
Rentang tanggal : 2022-01-03 s/d 2026-04-30
Jumlah komoditas: 21
Missing values  : 0

Tipe data:
Komoditas            object
tanggal      datetime64[ns]
harga               float64
sumber               object
dtype: object

Daftar komoditas:
  - Bawang Merah Ukuran Sedang
  - Bawang Putih Ukuran Sedang
  - Beras Kualitas Bawah I
  - Beras Kualitas Bawah II
  - Beras Kualitas Medium I
  - Beras Kualitas Medium II
  - Beras Kualitas Super I
  - Beras Kualitas Super II
  - Cabai Merah Besar
  - Cabai Merah Keriting
  - Cabai Rawit Hijau
  - Cabai Rawit Merah
  - Daging Ayam Ras Segar
  - Daging Sapi Kualitas 1
  - Daging Sapi Kualitas 2
  - Gula Pasir Kualitas Premium
  - Gula Pasir Lokal
  - Minyak Goreng Curah
  - Minyak Goreng Kemasan Bermerk 1
  - Minyak Goreng Kemasan Bermerk 2
  - Telur Ayam Ras Segar


In [96]:
df_pihps.head(10)

,Komoditas,tanggal,harga,sumber
0,Bawang Merah Ukuran Sedang,2022-01-03,30200.0,PIHPS
1,Bawang Merah Ukuran Sedang,2022-01-04,30150.0,PIHPS
2,Bawang Merah Ukuran Sedang,2022-01-05,30300.0,PIHPS
3,Bawang Merah Ukuran Sedang,2022-01-06,30300.0,PIHPS
4,Bawang Merah Ukuran Sedang,2022-01-07,30400.0,PIHPS
5,Bawang Merah Ukuran Sedang,2022-01-10,30650.0,PIHPS
6,Bawang Merah Ukuran Sedang,2022-01-11,30600.0,PIHPS
7,Bawang Merah Ukuran Sedang,2022-01-12,30550.0,PIHPS
8,Bawang Merah Ukuran Sedang,2022-01-13,30650.0,PIHPS
9,Bawang Merah Ukuran Sedang,2022-01-14,30750.0,PIHPS


### 3.5 Deteksi Outlier (IQR Method)

In [97]:
def flag_outliers_iqr(df, group_col='Komoditas', value_col='harga', factor=1.5):
    def iqr_bounds(x):
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        return (x < q1 - factor * iqr) | (x > q3 + factor * iqr)
    return df.groupby(group_col)[value_col].transform(iqr_bounds)

df_pihps['is_outlier'] = flag_outliers_iqr(df_pihps)

# Ringkasan outlier per komoditas
outlier_summary = (
    df_pihps.groupby('Komoditas')['is_outlier']
    .agg(total='count', n_outlier='sum')
    .assign(pct=lambda x: (x['n_outlier'] / x['total'] * 100).round(2))
    .sort_values('n_outlier', ascending=False)
)
print('=== Ringkasan Outlier per Komoditas ===')
print(outlier_summary.to_string())
print(f'\nTotal outlier : {df_pihps["is_outlier"].sum()} dari {len(df_pihps):,} baris ({df_pihps["is_outlier"].mean()*100:.2f}%)')
print('\nCatatan: outlier TIDAK dihapus, lonjakan harga saat Lebaran/krisis adalah sinyal valid untuk model.')

=== Ringkasan Outlier per Komoditas ===
                                 total  n_outlier    pct
Komoditas                                               
Daging Sapi Kualitas 1            1129        149  13.20
Daging Sapi Kualitas 2            1129        124  10.98
Minyak Goreng Kemasan Bermerk 1   1129         83   7.35
Minyak Goreng Kemasan Bermerk 2   1129         77   6.82
Telur Ayam Ras Segar              1129         60   5.31
Bawang Merah Ukuran Sedang        1129         43   3.81
Cabai Merah Besar                 1129         30   2.66
Cabai Merah Keriting              1129         19   1.68
Daging Ayam Ras Segar             1129         17   1.51
Cabai Rawit Hijau                 1129         11   0.97
Cabai Rawit Merah                 1129          3   0.27
Beras Kualitas Super I            1129          0   0.00
Beras Kualitas Super II           1129          0   0.00
Bawang Putih Ukuran Sedang        1129          0   0.00
Beras Kualitas Medium II          1129          

### 3.6 Cleaning SP2KP

In [98]:
df_sp2kp = raw_sp2kp.copy()

# Rename kolom pertama (artefak BOM UTF-16 terbaca sebagai 'cekNo')
df_sp2kp.rename(columns={df_sp2kp.columns[0]: 'No'}, inplace=True)

# Pisahkan kolom identitas dan kolom tanggal SEBELUM cleaning nama kolom
# agar format tanggal (02/01/2026) tidak ikut terhapus slash-nya
ID_COL_COUNT = 6
id_cols_raw   = df_sp2kp.columns[:ID_COL_COUNT].tolist()
date_cols_raw = df_sp2kp.columns[ID_COL_COUNT:].tolist()

# Bersihkan hanya kolom identitas
id_cols_clean = (
    pd.Index(id_cols_raw)
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('/', '', regex=False)
    .tolist()
)

# Kolom tanggal: cukup strip spasi, jangan hapus slash
date_cols_clean = pd.Index(date_cols_raw).str.strip().tolist()

# Terapkan nama kolom baru
df_sp2kp.columns = id_cols_clean + date_cols_clean
id_cols   = id_cols_clean
date_cols = date_cols_clean

print(f'Kolom identitas : {id_cols}')
print(f'Kolom tanggal   : {len(date_cols)} kolom')
print(f'Tanggal pertama : {date_cols[0]}')
print(f'Tanggal terakhir: {date_cols[-1]}')

Kolom identitas : ['no', 'kode_wilayah', 'provinsi', 'kabupaten_kota', 'komoditas', 'hetha']
Kolom tanggal   : 59 kolom
Tanggal pertama : 02/01/2026
Tanggal terakhir: 30/04/2026


In [99]:
# Melt: wide -> long
df_sp2kp_long = df_sp2kp.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name='tanggal',
    value_name='harga'
)

# Konversi tanggal
df_sp2kp_long['tanggal'] = pd.to_datetime(df_sp2kp_long['tanggal'], format='%d/%m/%Y', errors='coerce')

# Pandas sudah membaca nilai sebagai float (45.000 -> 45.0)
# Nilai SP2KP dalam satuan ribuan rupiah -> kalikan 1000 agar setara dengan PIHPS
df_sp2kp_long['harga'] = pd.to_numeric(df_sp2kp_long['harga'], errors='coerce') * 1000

# Hapus baris tanpa harga atau tanggal
df_sp2kp_long = df_sp2kp_long.dropna(subset=['harga', 'tanggal'])

# Tandai sumber data
df_sp2kp_long['sumber'] = 'SP2KP'

# Urutkan
df_sp2kp_long = df_sp2kp_long.sort_values([id_cols[3], id_cols[4], 'tanggal']).reset_index(drop=True)

print(f'SP2KP setelah cleaning: {df_sp2kp_long.shape[0]:,} baris x {df_sp2kp_long.shape[1]} kolom')
print(f'Missing values        : {df_sp2kp_long["harga"].isna().sum()}')
print()
print('Validasi range harga per komoditas (Rp):')
stats = df_sp2kp_long.groupby('komoditas')['harga'].agg(['min','max','mean']).round(0).astype(int)
print(stats.to_string())
print()
df_sp2kp_long.head()

SP2KP setelah cleaning: 456,590 baris x 9 kolom
Missing values        : 0

Validasi range harga per komoditas (Rp):
                                       min     max    mean
komoditas                                                 
Bawang Merah                         18000  100000   43331
Bawang Putih Honan                   20000  100000   39194
Beras Medium                         10500   50000   14290
Beras Premium                        13050   60000   16051
Cabai Merah Besar                    10000  190000   43583
Cabai Merah Keriting                 10000  200000   43458
Cabai Rawit Merah                    15000  200000   64649
Daging Ayam Ras                      20000  100000   40396
Daging Sapi Paha Belakang            90000  200000  138413
Garam Halus                           5000   50000   11810
Gula Pasir Curah                     13000   45000   18483
Ikan Kembung                         15000  100000   45112
Minyak Goreng Sawit Curah            13000   35000   18283

,no,kode_wilayah,provinsi,kabupaten_kota,komoditas,hetha,tanggal,harga,sumber
0,62.0,1105,Aceh,Kab. Aceh Barat,Bawang Merah,41.5,2026-01-02,50000.0,SP2KP
1,62.0,1105,Aceh,Kab. Aceh Barat,Bawang Merah,41.5,2026-01-05,48000.0,SP2KP
2,62.0,1105,Aceh,Kab. Aceh Barat,Bawang Merah,41.5,2026-01-06,48000.0,SP2KP
3,62.0,1105,Aceh,Kab. Aceh Barat,Bawang Merah,41.5,2026-01-07,48000.0,SP2KP
4,62.0,1105,Aceh,Kab. Aceh Barat,Bawang Merah,41.5,2026-01-08,48000.0,SP2KP


### 3.7 Simpan Dataset Hasil Cleaning

In [100]:
os.makedirs('data/cleaned', exist_ok=True)

df_pihps.to_csv('data/cleaned/pihps_cleaned.csv', index=False)
df_sp2kp_long.to_csv('data/cleaned/sp2kp_cleaned.csv', index=False)

print('Dataset tersimpan:')
print(f'  data/cleaned/pihps_cleaned.csv  | {df_pihps.shape[0]:,} baris x {df_pihps.shape[1]} kolom')
print(f'  data/cleaned/sp2kp_cleaned.csv  | {df_sp2kp_long.shape[0]:,} baris x {df_sp2kp_long.shape[1]} kolom')
print()
print('Catatan untuk AI Engineer:')
print('  - Kolom "tanggal" tersimpan sebagai string di CSV.')
print('    Saat load gunakan: pd.read_csv(..., parse_dates=[\"tanggal\"])')
print('  - Kolom "is_outlier" (bool) menandai harga anomali, jangan dihapus,')
print('    bisa dijadikan fitur tambahan untuk model.')

Dataset tersimpan:
  data/cleaned/pihps_cleaned.csv  | 23,709 baris x 5 kolom
  data/cleaned/sp2kp_cleaned.csv  | 456,590 baris x 9 kolom

Catatan untuk AI Engineer:
  - Kolom "tanggal" tersimpan sebagai string di CSV.
    Saat load gunakan: pd.read_csv(..., parse_dates=["tanggal"])
  - Kolom "is_outlier" (bool) menandai harga anomali, jangan dihapus,
    bisa dijadikan fitur tambahan untuk model.


---
## 4. Ringkasan Data Wrangling

| Tahap | Kegiatan | Status |
|-------|----------|--------|
| Gathering | Load 5 file PIHPS (2022-2026) + SP2KP | done |
| Assessing | Cek shape, tipe data, missing values, konsistensi | done |
| Cleaning | Filter header romawi, strip whitespace, konversi tipe | done |
| Cleaning | Replace `-` -> NaN, interpolasi linear | done |
| Cleaning | Melt wide -> long, konversi tanggal & harga | done |
| Cleaning | Gabungkan 5 tahun, simpan ke CSV | done |